<a href="https://colab.research.google.com/github/TAUforPython/LLM_AI_agents/blob/main/lesson-8_Different_RAG_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medical Appointment Assistant with RAG Transformations


This notebook demonstrates RAG transformations for a medical appointment assistant:
1. Query Rewriting - Convert conversational queries to search-friendly format
2. Query Decomposition - Break complex queries into sub-queries
3. HyDE - Generate hypothetical documents to improve search relevance


Let's show how each RAG method handled the complex medical question about specialist referrals and medical record access:

##Query Rewriting:

Original Query: "I need to understand the process for obtaining specialist referrals, including any insurance pre-authorization requirements, and also clarify the policy regarding access to my medical records, especially if I need electronic copies or if a family member requires access."
Rewritten Query: "What is the process for obtaining specialist referrals in healthcare, including insurance pre-authorization requirements? Also, what are the policies for accessing medical records, such as obtaining electronic copies or granting family member access?"
Analysis: Query rewriting successfully transformed the conversational query into a more concise and keyword-rich question. The response generated from this rewritten query was quite comprehensive, covering both specialist referrals and medical records access, including details on PCP approval, pre-authorization, validity periods, request processes, electronic access, and fees. It retrieved relevant documents such as 'doc_7' (Specialist Referral Process), 'doc_2' (Insurance Verification), and 'doc_9' (Medical Records Access). It also pulled 'doc_5' (Lab Results), which while related to records, might not be the most direct fit for the general records access policy.

##Query Decomposition:

Original Query: Same as above.
Sub-Queries: The original query was broken down into distinct sub-queries: 'Specialist Referral Process', 'Insurance Pre-Authorization Requirements', 'Medical Records Access Policy', and 'Family Member Access to Medical Records'.
Analysis: This method provided the most structured and detailed answer. By breaking the complex query into individual components, it allowed for more targeted retrieval of information for each specific aspect. The response was organized with clear headings for Specialist Referral Process, Insurance Verification Process, Access to Medical Records, and Patient Privacy Rights. It even included a helpful "Summary Checklist." This approach ensures that all parts of the complex question are addressed thoroughly, potentially leading to a more complete and accurate answer, as seen by the broader set of source documents including 'doc_3' (Telemedicine Policy, likely for general insurance context) and 'doc_10' (Patient Privacy Rights).

##HyDE Search (Truncated Output):

Expected Behavior: HyDE would first generate a hypothetical document that perfectly answers the complex query. This hypothetical document's embedding would then be used to search for the most semantically similar actual documents in the vector store. This typically helps in cases where the original query's keywords might not directly match the document's wording but its intent is clear.
Comparison: For a highly complex, multi-faceted query, a single hypothetical document might still struggle to capture all nuances as effectively as explicit query decomposition. However, it would likely outperform a standard search by providing a richer context for retrieval.
Standard Search (Truncated Output):

Expected Behavior: A standard search would take the original, lengthy, and multi-part query and perform a direct similarity search against the vector store. It often struggles with complex queries because it tries to find a single set of documents that match the entire query, potentially leading to less focused and less comprehensive results compared to the other techniques.
Comparison: For this type of complex query, standard search is generally the least effective as it might dilute the intent across multiple topics and fail to retrieve documents that address all distinct aspects accurately.
Conclusion: For a complex medical question like the one provided, Query Decomposition appears to be the most effective strategy. It explicitly breaks down the user's intent into manageable sub-queries, leading to more precise information retrieval and a highly structured, comprehensive answer. Query Rewriting is also beneficial by improving searchability, but decomposition offers a more granular approach to multi-part questions. Both HyDE and standard search would likely struggle more with the inherent complexity of a multi-topic query compared to decomposition.

In [ ]:
!pip install langchain_mistralai -q

In [ ]:
# Install required packages
!pip install langchain langchain-openai langchain-community transformers faiss-cpu -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
!pip install langchain-text-splitters -q


In [15]:
!pip install langchain_classic -q

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [17]:
from langchain_classic.schema import Document
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate

In [25]:
import os
from typing import List, Dict, Any
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import re
import tenacity

In [19]:

MODEL_NAME = 'mistral-small-latest' # or "mistral-large-latest"

import os
from google.colab import userdata

os.environ["MISTRAL_API_KEY"] = userdata.get("Mistral_API")

from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model=MODEL_NAME,
    temperature=0,
    max_retries=2,
)

# Database

In [26]:
from langchain_mistralai import MistralAIEmbeddings
from langchain_classic.schema import Document
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
import httpx # Import httpx to catch HTTPStatusError

# Medical knowledge base - simulated with medical policies, procedures, and FAQs
MEDICAL_KNOWLEDGE_BASE = [
    """
    Medical Appointment Cancellation Policy
    Cancellation must be made at least 24 hours before the scheduled appointment.
    Cancellations made less than 24 hours prior to the appointment will incur a $25 fee.
    Emergency situations (hospitalization, accident) are exempt from cancellation fees with proper documentation.
    Cancellation requests must be made via phone call or through the patient portal.
    """,
    """
    Prescription Refill Procedure
    Prescription refills require 48-hour advance notice.
    Patients must submit refill requests through the patient portal or by calling the pharmacy.
    Controlled substances require an in-person visit every 90 days for renewal.
    Chronic medications may be prescribed for up to 90 days at a time.
    Emergency refill requests outside normal hours may be handled by on-call staff for established patients.
    """,
    """
    Insurance Verification Process
    All patients must provide valid insurance information at registration.
    The clinic verifies coverage eligibility before each appointment.
    If insurance coverage lapses, patients are responsible for the full cost of services rendered.
    Prior authorization is required for specialist referrals and diagnostic procedures.
    Out-of-network providers may require pre-payment with subsequent reimbursement claims.
    """,
    """
    Telemedicine Consultation Policy
    Telemedicine appointments require stable internet connection and compatible device.
    Coverage varies by insurance provider - patients should verify telehealth benefits before scheduling.
    Certain conditions require in-person evaluation and cannot be treated remotely.
    Technical difficulties during consultation may necessitate rescheduling.
    Telemedicine visits are billed similarly to in-person appointments.
    Prescriptions issued via telemedicine follow standard refill protocols.
    """,
    """
    Follow-up Appointment Scheduling
    Follow-up appointments are typically scheduled within 2-4 weeks of the initial consultation depending on condition severity.
    Urgent follow-ups (post-surgery, critical lab results) are prioritized for scheduling within 7 days.
    Chronic disease management requires regular appointments every 3-6 months.
    Patients are contacted via preferred communication method (phone or email) to schedule follow-ups.
    Reminder notifications are sent 48 hours before scheduled appointments.
    """,
    """
    Lab Results Access Policy
    Lab results are available through the patient portal within 2-5 business days.
    Critical abnormal results are communicated directly by phone within 24 hours.
    Patients should review results with their healthcare provider to discuss implications and treatment options.
    Older results (over 1 year) may require a request form for retrieval.
    Lab reports are retained for 7 years per medical record regulations.
    Patients may request copies of results for external providers.
    """,
    """
    Vaccination Requirements
    Routine vaccinations follow CDC guidelines and are recommended based on age, medical history, and risk factors.
    Travel vaccinations require advance planning (2-4 weeks before travel).
    Some vaccines require multiple doses over several weeks.
    Vaccination records should be updated annually.
    School and employment requirements may mandate specific immunizations.
    Contraindications and allergies must be disclosed before vaccination administration.
    """,
    """
    Specialist Referral Process
    Specialist referrals require primary care physician approval and insurance pre-authorization.
    Referral validity period is typically 90 days from issuance.
    Urgent referrals are processed within 1-2 business days.
    Routine referrals may take up to 2 weeks for approval.
    Patients are responsible for scheduling with specialists within referral validity period.
    Copayments and deductibles apply per insurance plan for specialist visits.
    """,
    """
    Payment and Billing Policies
    Co-payments are collected at the time of service.
    Outstanding balances become due within 30 days of billing statement.
    Late payments may incur service fees.
    Payment plans are available for balances exceeding $500.
    Insurance claims are submitted electronically.
    Denied claims are reviewed for appeal opportunities.
    Patients without insurance receive discounted self-pay rates upon request.
    """,
    """
    Medical Records Access
    Patients have the right to access their complete medical records.
    Requests require written authorization and valid identification.
    Records are provided within 30 days of request.
    Electronic records are available through the patient portal.
    Copies of records may incur a nominal fee for printing and processing.
    Legal requests (court orders, disability evaluations) follow expedited processing.
    Next of kin access requires power of attorney or legal guardianship documentation.
    """,
    """
    Patient Privacy Rights
    Patient information is protected under HIPAA regulations.
    Consent is required for information sharing with external providers.
    Patients may restrict information sharing for specific treatments or family members.
    Confidentiality exceptions include court orders, suspected abuse, or public health reporting requirements.
    Patients may request restrictions on electronic health record access by staff members.
    Breach notification occurs within 60 days of discovery if patient information is compromised.
    """,
    """
    Chronic Disease Management Program
    Diabetes, hypertension, and other chronic conditions require structured management plans.
    Regular monitoring appointments are scheduled every 3-6 months.
    Patient education sessions cover self-management techniques and lifestyle modifications.
    Medication adherence support includes reminder systems and pharmacy coordination.
    Annual comprehensive evaluations assess treatment effectiveness.
    Coordinated care involves specialists and allied health professionals as needed.
    """
]

# Create vector store from the knowledge base
documents = [Document(page_content=text, metadata={"source": f"doc_{i}"}) for i, text in enumerate(MEDICAL_KNOWLEDGE_BASE)]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)
embeddings = MistralAIEmbeddings() # Use MistralAIEmbeddings
vector_store = FAISS.from_documents(texts, embeddings)

# Initialize the LLM (using the 'llm' already defined in the previous cell)

print("Medical knowledge base loaded and vector store created!")
print(f"Knowledge base contains {len(MEDICAL_KNOWLEDGE_BASE)} documents")
print(f"Vector store has {len(vector_store.docstore._dict)} text chunks")

# 1. Query Rewriting
class QueryRewriter:
    """
    Rewrites conversational queries to search-friendly format using LLM
    """
    def __init__(self, llm):
        self.llm = llm
        self.rewrite_prompt = PromptTemplate(
            input_variables=["chat_history", "user_query"],
            template="""
            Given the chat history and a user query, rewrite the user query to be more specific and search-friendly.
            The rewritten query should contain keywords that would help retrieve relevant documents from a medical knowledge base.

            Chat History: {chat_history}

            Original Query: {user_query}

            Rewritten Query: """
        )

    @retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5), retry=retry_if_exception_type(httpx.HTTPStatusError))
    def rewrite_query(self, chat_history: str, user_query: str) -> str:
        """
        Rewrite the user query to be more search-friendly
        """
        chain = self.rewrite_prompt | self.llm
        response = chain.invoke({
            "chat_history": chat_history,
            "user_query": user_query
        })
        return response.content.strip()

# 2. Query Decomposition
class QueryDecomposer:
    """
    Decomposes complex queries into simpler sub-queries for parallel search
    """
    def __init__(self, llm):
        self.llm = llm
        self.decompose_prompt = PromptTemplate(
            input_variables=["query"],
            template="""
            Decompose the following query into 2-4 simpler sub-queries that can be searched independently.
            Each sub-query should focus on a specific aspect of the original query.

            Original Query: {query}

            Sub-queries:
            1.
            2.
            3.
            4. """
        )

    @retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5), retry=retry_if_exception_type(httpx.HTTPStatusError))
    def decompose_query(self, query: str) -> List[str]:
        """
        Decompose a complex query into simpler sub-queries
        """
        chain = self.decompose_prompt | self.llm
        response = chain.invoke({"query": query})
        result = response.content.strip()

        # Extract sub-queries
        sub_queries = []
        for line in result.split('\n'):
            line = line.strip()
            if line.startswith(('1.', '2.', '3.', '4.')) and len(line) > 2:
                sub_query = line[2:].strip()  # Remove the number and period
                if sub_query:
                    sub_queries.append(sub_query)

        return sub_queries

# 3. HyDE (Hypothetical Document Embeddings)
class HyDEGenerator:
    """
    Generates hypothetical documents for a query to improve search relevance
    """
    def __init__(self, llm):
        self.llm = llm
        self.hyde_prompt = PromptTemplate(
            input_variables=["query"],
            template="""
            Write a hypothetical document that would perfectly answer the following query.
            The document should be detailed and comprehensive, containing all the information needed to address the query.

            Query: {query}

            Hypothetical Document: """
        )

    @retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5), retry=retry_if_exception_type(httpx.HTTPStatusError))
    def generate_hypothetical_document(self, query: str) -> str:
        """
        Generate a hypothetical document that would answer the query
        """
        chain = self.hyde_prompt | self.llm
        response = chain.invoke({"query": query})
        return response.content.strip()

# Main RAG system combining all techniques
class MedicalRAGSystem:
    """
    Medical RAG system with Query Rewriting, Decomposition, and HyDE
    """
    def __init__(self, vector_store, llm):
        self.vector_store = vector_store
        self.llm = llm
        self.query_rewriter = QueryRewriter(llm)
        self.query_decomposer = QueryDecomposer(llm)
        self.hyde_generator = HyDEGenerator(llm)

        # Standard QA chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vector_store.as_retriever(),
            return_source_documents=True
        )

    def query_rewrite_search(self, chat_history: str, query: str) -> Dict[str, Any]:
        """
        Use query rewriting before searching
        """
        # Rewrite the query
        rewritten_query = self.query_rewriter.rewrite_query(chat_history, query)
        print(f"Original Query: {query}")
        print(f"Rewritten Query: {rewritten_query}")

        # Search with rewritten query
        result = self.qa_chain.invoke(rewritten_query)
        return {
            "original_query": query,
            "rewritten_query": rewritten_query,
            "response": result["result"],
            "source_docs": result.get("source_documents", [])
        }

    def query_decomposition_search(self, query: str) -> Dict[str, Any]:
        """
        Use query decomposition with parallel search
        """
        # Decompose the query
        sub_queries = self.query_decomposer.decompose_query(query)
        print(f"Original Query: {query}")
        print(f"Sub-Queries: {sub_queries}")

        # Search for each sub-query
        all_docs = []
        for sub_query in sub_queries:
            docs = self.vector_store.similarity_search(sub_query, k=2)
            all_docs.extend(docs)

        # Remove duplicate documents
        unique_docs = []
        seen_content = set()
        for doc in all_docs:
            if doc.page_content not in seen_content:
                unique_docs.append(doc)
                seen_content.add(doc.page_content)

        # Create context from all retrieved documents
        context = "\n\n".join([doc.page_content for doc in unique_docs])

        # Generate response based on combined context
        prompt = f"""
        Based on the following medical information, answer the user's query: {query}

        Medical Information:
        {context}

        Answer: """

        response = self.llm.invoke(prompt)

        return {
            "original_query": query,
            "sub_queries": sub_queries,
            "response": response.content,
            "source_docs": unique_docs
        }

    def hyde_search(self, query: str) -> Dict[str, Any]:
        """
        Use HyDE to generate hypothetical document then search
        """
        # Generate hypothetical document
        hypothetical_doc = self.hyde_generator.generate_hypothetical_document(query)
        print(f"Original Query: {query}")
        print(f"Hypothetical Document: {hypothetical_doc[:200]}...")

        # Search using the hypothetical document as the query
        docs = self.vector_store.similarity_search(hypothetical_doc, k=4)

        # Create context from retrieved documents
        context = "\n\n".join([doc.page_content for doc in docs])

        # Generate final response
        prompt = f"""
        Based on the following medical information, answer the user's query: {query}

        Medical Information:
        {context}

        Answer: """

        response = self.llm.invoke(prompt)

        return {
            "original_query": query,
            "hypothetical_document": hypothetical_doc,
            "response": response.content,
            "source_docs": docs
        }

# Initialize the system
medical_rag_system = MedicalRAGSystem(vector_store, llm)

# Example usage of each technique
print("="*60)
print("EXAMPLE 1: QUERY REWRITING")
print("="*60)

chat_history = "User: I need to cancel my appointment for next Tuesday. Agent: Sure, I can help with that. What is your appointment ID?"
query1 = "I have an emergency and need to cancel right away. Will I be charged?"

result1 = medical_rag_system.query_rewrite_search(chat_history, query1)
print(f"Response: {result1['response']}\n")

print("="*60)
print("EXAMPLE 2: QUERY DECOMPOSITION")
print("="*60)

query2 = "I need to know about prescription refills and also the policy for canceling appointments"

result2 = medical_rag_system.query_decomposition_search(query2)
print(f"Response: {result2['response']}\n")

print("="*60)
print("EXAMPLE 3: HYDE (HYPOTHETICAL DOCUMENT EMBEDDINGS)")
print("="*60)

query3 = "How do I access my lab results?"

result3 = medical_rag_system.hyde_search(query3)
print(f"Response: {result3['response']}\n")

# Compare with standard search
print("="*60)
print("COMPARISON: STANDARD SEARCH vs RAG TECHNIQUES")
print("="*60)

standard_result = medical_rag_system.qa_chain.invoke(query3)
print(f"Standard Search Response: {standard_result['result']}\n")

print(f"HyDE Response: {result3['response']}\n")

print("Notice how HyDE generates a more comprehensive and relevant response by first creating a hypothetical document that matches the query intent.")

Medical knowledge base loaded and vector store created!
Knowledge base contains 12 documents
Vector store has 18 text chunks
EXAMPLE 1: QUERY REWRITING
Original Query: I have an emergency and need to cancel right away. Will I be charged?
Rewritten Query: **Rewritten Query:** *"What are the cancellation policies and fees for emergency appointment cancellations? Will I be charged if I cancel my appointment immediately?"*

**Keywords added for search-friendliness:**
- Cancellation policies
- Fees
- Emergency appointment
- Immediate cancellation
- Charges

This version ensures the query is specific to medical appointment policies while retaining urgency and clarity.
Response: According to the **Medical Appointment Cancellation Policy**:

- **Emergency situations** (e.g., hospitalization, accident) are **exempt from cancellation fees** if proper documentation is provided.
- **Cancellations made less than 24 hours before the appointment** (without an emergency) incur a **$25 fee**.

If you c

In [22]:
def test_rag_system(user_query: str, chat_history: str = ""):
    """
    Tests the MedicalRAGSystem with a custom user query and displays results from all techniques.
    """
    print("\n" + "="*60)
    print(f"TESTING CUSTOM QUERY: {user_query}")
    print("="*60)

    # 1. Query Rewriting Test
    print("\n" + "-"*20 + " Query Rewriting " + "-"*20)
    qr_result = medical_rag_system.query_rewrite_search(chat_history, user_query)
    print(f"Response: {qr_result['response']}\n")
    print(f"Source Documents: {[doc.metadata['source'] for doc in qr_result['source_docs']]}")

    # 2. Query Decomposition Test
    print("\n" + "-"*20 + " Query Decomposition " + "-"*20)
    qd_result = medical_rag_system.query_decomposition_search(user_query)
    print(f"Response: {qd_result['response']}\n")
    print(f"Source Documents: {[doc.metadata['source'] for doc in qd_result['source_docs']]}")

    # 3. HyDE Test
    print("\n" + "-"*20 + " HyDE Search " + "-"*20)
    hyde_result = medical_rag_system.hyde_search(user_query)
    print(f"Response: {hyde_result['response']}\n")
    print(f"Source Documents: {[doc.metadata['source'] for doc in hyde_result['source_docs']]}")

    # Compare with standard search
    print("\n" + "-"*20 + " Standard Search " + "-"*20)
    standard_result = medical_rag_system.qa_chain.invoke(user_query)
    print(f"Response: {standard_result['result']}\n")
    print(f"Source Documents: {[doc.metadata['source'] for doc in standard_result['source_documents']]}")

    print("\n" + "="*60)
    print("END OF CUSTOM QUERY TEST")
    print("="*60)


# Try it out with a custom query!

Use the `test_rag_system` function to experiment with different medical-related questions.

In [27]:
# Example custom query
custom_query = "What do I need to know about getting my child vaccinated before school?"
test_rag_system(custom_query)

# Another example with chat history for query rewriting
custom_query_2 = "How much will it cost me if I don't have insurance?"
custom_chat_history_2 = "User: I need to see a doctor but I don't have insurance. Agent: We have options for patients without insurance."
test_rag_system(custom_query_2, custom_chat_history_2)


TESTING CUSTOM QUERY: What do I need to know about getting my child vaccinated before school?

-------------------- Query Rewriting --------------------
Original Query: What do I need to know about getting my child vaccinated before school?
Rewritten Query: **Rewritten Query:** *"Essential information about required childhood vaccinations for school enrollment, including recommended vaccines, schedules, safety, and legal requirements in the U.S. (or [specific country/region if known])"*
Response: Here’s the essential information about required childhood vaccinations for school enrollment in the **U.S.**, based on the provided context and general knowledge:

### **1. Recommended Vaccines for School Enrollment**
The **CDC** (Centers for Disease Control and Prevention) provides routine vaccination schedules for children. Common required vaccines for school entry include:
- **Diphtheria, Tetanus, Pertussis (DTaP/Tdap)**
- **Measles, Mumps, Rubella (MMR)**
- **Polio (IPV)**
- **Hepatitis B

In [29]:
# Compare RAG technique outputs for a complex medical question
complex_medical_query = "I need to understand the process for obtaining specialist referrals, including any insurance pre-authorization requirements, and also clarify the policy regarding access to my medical records, especially if I need electronic copies or if a family member requires access."
test_rag_system(complex_medical_query)


TESTING CUSTOM QUERY: I need to understand the process for obtaining specialist referrals, including any insurance pre-authorization requirements, and also clarify the policy regarding access to my medical records, especially if I need electronic copies or if a family member requires access.

-------------------- Query Rewriting --------------------
Original Query: I need to understand the process for obtaining specialist referrals, including any insurance pre-authorization requirements, and also clarify the policy regarding access to my medical records, especially if I need electronic copies or if a family member requires access.
Rewritten Query: Here’s a more specific and search-friendly rewritten query with medical and insurance-related keywords:

**"What is the process for obtaining specialist referrals in healthcare, including insurance pre-authorization requirements? Also, what are the policies for accessing medical records, such as obtaining electronic copies or granting family